# 01 - Setup, data, baseline generation, SFT  (Tasks 1, 2, 3)

What this notebook does (about 2 h on T4 x2):
1. Clones the repo, installs dependencies without touching PyTorch, checks the 2 GPUs and versions (Task 1).
2. Runs the public + own adapter tests and a 3-minute tiny-model smoke test of the whole pipeline.
3. Downloads and filters the data: 30k/1k UltraChat SFT, 15k/1k HH-RLHF, 15k/1k UltraFeedback, AlpacaEval-300 (seed 42).
4. Baseline generations from `Qwen/Qwen2.5-0.5B` on **cuda:1 in the background** (Task 2) while
5. SFT trains on **cuda:0** (Task 3). Best + final checkpoints go to `outputs/sft/`.

Output used by notebooks 02 and 03: `outputs/data/`, `outputs/sft/{best,final}`, `outputs/generations/base_*.json`.

**Kaggle settings for this notebook:** Accelerator = **GPU T4 x2**, Internet = **On** (needs a phone-verified account),
Persistence = Files only (optional). Edit `REPO_URL` below, then **Save Version -> Save & Run All (Commit)** so the
run continues in the background and all files in `/kaggle/working` are kept as this version's output.

In [ ]:
# ---- EDIT THIS: your GitHub fork of the starter repo with this code pushed to it ----
REPO_URL = "https://github.com/Ibrahim2424777/CS5326-PA2.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/cs5326-pa2"

RUN_SMOKE_TEST = True      # ~3 min end-to-end check of every script on a tiny random model
RUN_BASELINE_HH = True     # also generate base-model outputs on the HH eval set (for your own analysis)

In [ ]:
import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
from smalign.notebook import sh, start_bg, wait_bg, find_input, copy_run_metadata, zip_dir, disk_report

In [ ]:
# Task 1: install deps WITHOUT reinstalling Kaggle's CUDA-matched PyTorch (manual 5.2)
sh("python -m pip uninstall -y -q torchao || true")
sh('python -m pip install -q "transformers>=4.56.2,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml '
   'sentencepiece safetensors pytest matplotlib')
sh("python -m pip install -q -e . --no-deps")

In [ ]:
# Task 1: confirm two T4 GPUs and print package versions
sh("nvidia-smi")
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if torch.cuda.device_count() < 2:
    print("WARNING: expected GPU T4 x2 (Settings -> Accelerator). Scripts will fall back to one GPU but run slower.")
disk_report()

## Adapter tests (public smoke tests + our own edge cases)

In [ ]:
sh("PYTHONPATH=. python -m pytest -q tests/")

In [ ]:
if RUN_SMOKE_TEST:
    sh("python scripts/smoke_test_pipeline.py --work_dir /tmp/smoke", log_path="outputs/logs/smoke.log")

## Data preparation (fixed, seeded subsets; stats in `outputs/data/data_stats.json`)

In [ ]:
sh("python scripts/prepare_data.py --out_dir outputs/data", log_path="outputs/logs/prepare_data.log")
sh("wc -l outputs/data/*.jsonl && cat outputs/data/data_stats.json")

## Task 2 - baseline generation
First a 3-prompt sanity check (prompt format, EOS, length), then the full 300-prompt AlpacaEval subset in the
background on cuda:1 so it overlaps with SFT training. Decoding: greedy, repetition_penalty 1.1, 512 new tokens.

In [ ]:
BASE = "Qwen/Qwen2.5-0.5B"
sh(f"python scripts/generate.py --model_path {BASE} --generator qwen2.5-0.5b-base "
   f"--eval_set eval_sets/alpaca_eval_300_seed42.json --output_path /tmp/sanity_base.json "
   f"--limit 3 --print_samples --device cuda:1")

In [ ]:
base_cmds = [f"python scripts/generate.py --model_path {BASE} --generator qwen2.5-0.5b-base "
             f"--eval_set eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/base_alpaca.json --device cuda:1"]
if RUN_BASELINE_HH:
    base_cmds.append(f"python scripts/generate.py --model_path {BASE} --generator qwen2.5-0.5b-base "
                     f"--eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/base_hh.json --device cuda:1")
baseline_job = start_bg(" && ".join(base_cmds), "outputs/logs/baseline_generation.log")

## Task 3 - SFT (own training loop, `scripts/train_sft.py`, config `configs/sft.yaml`)
30k examples, batch 2 x accum 16 = 32, lr 2e-5, 3% warmup + cosine, wd 0.1, clip 1.0, fp16 autocast + fp32 weights,
validation every 100 steps, best + final checkpoints. If you hit CUDA OOM, add `gradient_checkpointing=true` to `--set`.

In [ ]:
sh("python scripts/train_sft.py --config configs/sft.yaml --set device=cuda:0", log_path="outputs/logs/sft.log")

In [ ]:
wait_bg(baseline_job, poll_seconds=60)
sh("ls -la outputs/generations outputs/sft outputs/sft/best")

## Task 6 sanity check for the SFT checkpoint (3 prompts, printed)

In [ ]:
sh("python scripts/generate.py --model_path outputs/sft/best --generator sft "
   "--eval_set eval_sets/alpaca_eval_300_seed42.json --output_path /tmp/sanity_sft.json --limit 3 --print_samples")

## Plots + archive of the small artifacts (checkpoints stay as regular notebook output)

In [ ]:
sh("python scripts/plot_training.py --sft outputs/sft --out_dir outputs/report_assets")
from IPython.display import Image, display
display(Image("outputs/report_assets/sft_loss.png"))
zip_dir("outputs", "/kaggle/working/sft_run_outputs.zip")
disk_report()
print("DONE. Save this version; attach its output to notebooks 02 and 03.")